# Potato Disease Detection using CNN
This notebook trains a convolutional neural network to classify potato leaf diseases.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight
from pathlib import Path

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

2026-02-10 00:35:21.614248: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/shashwat/Desktop/potato-disease-detection/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ModuleNotFoundError: No module named 'matplotlib'

## 1. Data Loading and Exploration

In [ ]:
# Define dataset path
DATASET_PATH = '/Users/shashwat/Desktop/Dataset'
IMAGE_SIZE = 256
BATCH_SIZE = 32
CLASS_NAMES = ['Early Blight', 'Late Blight', 'Healthy']

# Check dataset structure
print("Dataset structure:")
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    if os.path.exists(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{class_name}: {num_images} images")
    else:
        print(f"{class_name}: Path not found")

In [ ]:
# Load dataset using tf.keras.utils.image_dataset_from_directory
# This automatically creates train/val split
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    label_mode='int'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    label_mode='int'
)

print(f"Train batches: {len(train_dataset)}")
print(f"Validation batches: {len(val_dataset)}")

In [ ]:
# Visualize sample images from each class
plt.figure(figsize=(12, 8))
for images, labels in train_dataset.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(CLASS_NAMES[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate class weights to handle imbalance
# Extract all labels
all_labels = []
for images, labels in train_dataset:
    all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
print(f"Class distribution in training data:")
for i, class_name in enumerate(CLASS_NAMES):
    count = np.sum(all_labels == i)
    print(f"{class_name}: {count}")

# Compute class weights
class_weights_array = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print(f"\nClass weights: {class_weights}")

## 2. Data Preprocessing and Augmentation

In [ ]:
# Data augmentation for training data only
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
])

# Apply augmentation to training data
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Optimize datasets with caching and prefetching
train_dataset = train_dataset.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(tf.data.AUTOTUNE)

print("Data preprocessing complete!")

## 3. Build CNN Model

In [ ]:
# Build the CNN model
model = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1./255),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## 4. Train Model

In [ ]:
# Train the model with class weights
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    class_weight=class_weights
)

## 5. Model Evaluation

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on validation data
val_loss, val_accuracy = model.evaluate(val_dataset)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Get predictions on validation set for confusion matrix
y_true = []
y_pred = []

for images, labels in val_dataset:
    predictions = model.predict(images)
    y_pred.extend(np.argmax(predictions, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Show sample predictions
plt.figure(figsize=(14, 10))
for images, labels in val_dataset.take(2):
    predictions = model.predict(images)
    for i in range(min(8, len(images))):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        
        pred_class = np.argmax(predictions[i])
        true_class = labels[i].numpy()
        confidence = predictions[i][pred_class]
        
        color = 'green' if pred_class == true_class else 'red'
        plt.title(f"True: {CLASS_NAMES[true_class]}\nPred: {CLASS_NAMES[pred_class]} ({confidence:.2%})", color=color)
        plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
# Create saved_models directory if it doesn't exist
os.makedirs('saved_models', exist_ok=True)

# Save model
model.save('saved_models/potato_model.h5')
print("Model saved to saved_models/potato_model.h5")

# Also copy to backend directory
import shutil
backend_model_dir = '../backend/models'
os.makedirs(backend_model_dir, exist_ok=True)
shutil.copy('saved_models/potato_model.h5', f'{backend_model_dir}/potato_model.h5')
print(f"Model copied to {backend_model_dir}/potato_model.h5")